In [19]:
import os
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, silhouette_score

# 1) Evaluating generated embeddings using Silhouette score

In [18]:
def load_embeddings_and_labels(pkl_path):
    with open(pkl_path, "rb") as f:
        data = pickle.load(f)
    X = data["embeddings"]
    y = data["labels"]
    
    if hasattr(X, "cpu"):
        X = X.cpu().numpy()
    elif hasattr(X, "numpy"):
        X = X.numpy()
        
    if hasattr(y, "cpu"):
        y = y.cpu().numpy().ravel()
    elif hasattr(y, "numpy"):
        y = y.numpy().ravel()
        
    return X, y

def evaluate_all_experiments(experiments_dir_path="../experiments"):
    experiments_dir = Path(experiments_dir_path)
    records = []
    # iterating over the different configurations
    for config_dir in sorted(experiments_dir.iterdir()):
        if not config_dir.is_dir():
            continue
        config_name = config_dir.name
        seed_dirs = [d for d in config_dir.iterdir() if d.is_dir() and d.name.startswith("seed_")]
        if not seed_dirs:
            continue

        silhouette_scores = []
        for seed_dir in seed_dirs:
            train_pkl = seed_dir / "training_embeddings" / "embeddings.pkl"
            test_pkl = seed_dir / "test_embeddings" / "embeddings.pkl"
            if not (train_pkl.exists() and test_pkl.exists()):
                print(f"[WARNING] Missing embeddings in {seed_dir}")
                continue

            X_train, y_train = load_embeddings_and_labels(train_pkl)
            X_test, y_test = load_embeddings_and_labels(test_pkl)

            # Computing Silhouette Score on Test Set
            sil = silhouette_score(X_test, y_test, metric='euclidean')
            silhouette_scores.append(sil)
        # Mean and Stadev on different seeds (once per configuration, after the seeds loop)
        if silhouette_scores:
            sil_mean, sil_std = np.mean(silhouette_scores), np.std(silhouette_scores, ddof=1 if len(silhouette_scores) > 1 else 0)

            records.append({
                "Configuration": config_name,
                "Silhouette Score": f"{sil_mean:.4f} +- {sil_std:.4f}",
                "raw_sil_mean": sil_mean
            })

    df_results = pd.DataFrame(records)
    
    if not df_results.empty:
        df_results = df_results.sort_values(by="raw_sil_mean", ascending=False)
        df_results = df_results.drop(columns=["raw_sil_mean"])

    return df_results

In [ ]:
df_silhouette = evaluate_all_experiments(experiments_dir_path="../experiments")
df_silhouette

# 2) Fine Tuning SVC using Embeddings

In [21]:
from sklearn.svm import LinearSVC

def evaluate_all_experiments_with_svm(experiments_dir_path="../experiments"):
    experiments_dir = Path(experiments_dir_path)
    records = []

    for config_dir in sorted(experiments_dir.iterdir()):
        if not config_dir.is_dir():
            continue

        config_name = config_dir.name
        seed_dirs = [d for d in config_dir.iterdir() if d.is_dir() and d.name.startswith("seed_")]
        if not seed_dirs:
            continue

        svm_accuracies = []
        silhouette_scores = []
        for seed_dir in seed_dirs:
            train_pkl = seed_dir / "training_embeddings" / "embeddings.pkl"
            test_pkl = seed_dir / "test_embeddings" / "embeddings.pkl"

            if not (train_pkl.exists() and test_pkl.exists()):
                print(f"[WARNING] Missing embeddings in {seed_dir}")
                continue

            X_train, y_train = load_embeddings_and_labels(train_pkl)
            X_test, y_test = load_embeddings_and_labels(test_pkl)

            # Linear probing with SVM
            svm = LinearSVC(dual=False, max_iter=2000, random_state=42)
            svm.fit(X_train, y_train)
            y_pred_svm = svm.predict(X_test)
            svm_accuracies.append(accuracy_score(y_test, y_pred_svm))

            # Silhouette Score
            sil = silhouette_score(X_test, y_test, metric='euclidean')
            silhouette_scores.append(sil)

        if svm_accuracies:
            svm_mean, svm_std = np.mean(svm_accuracies), np.std(svm_accuracies, ddof=1 if len(svm_accuracies) > 1 else 0)
            sil_mean, sil_std = np.mean(silhouette_scores), np.std(silhouette_scores, ddof=1 if len(silhouette_scores) > 1 else 0)

            records.append({
                "Configuration": config_name,
                "Linear SVM Accuracy": f"{svm_mean*100:.2f}% +- {svm_std*100:.2f}%",
                "Silhouette Score": f"{sil_mean:.4f} +- {sil_std:.4f}",
                "_raw_svm_mean": svm_mean,
            })

    df_results = pd.DataFrame(records)
    
    # Ordering descending on SVM accuracy means
    if not df_results.empty:
        df_results = df_results.sort_values(by="_raw_svm_mean", ascending=False)
        df_results = df_results.drop(columns=["_raw_svm_mean"])

    return df_results

In [22]:
df_metrics_extended = evaluate_all_experiments_with_svm(experiments_dir_path="../experiments")
df_metrics_extended

,Configuration,Linear SVM Accuracy,Silhouette Score
28,4096_selu_1024_selu,82.01% +- 0.25%,-0.1313 +- 0.0046
24,4096_selu_1024_elu,81.90% +- 0.15%,-0.1386 +- 0.0010
4,4096_elu_1024_selu,81.80% +- 0.10%,-0.1313 +- 0.0009
29,4096_selu_1024_swish,81.66% +- 0.27%,-0.1276 +- 0.0036
0,4096_elu_1024_elu,81.54% +- 0.09%,-0.1344 +- 0.0022
25,4096_selu_1024_gelu,81.46% +- 0.11%,-0.1350 +- 0.0058
27,4096_selu_1024_relu,81.20% +- 0.06%,-0.1358 +- 0.0020
26,4096_selu_1024_lrelu,81.11% +- 0.26%,-0.1375 +- 0.0025
5,4096_elu_1024_swish,80.80% +- 0.24%,-0.1331 +- 0.0014
2,4096_elu_1024_lrelu,80.79% +- 0.12%,-0.1404 +- 0.0038
